# Limpiar cámaras y población
Objetivo: una tabla con una fila por cámara y una con la población de cada localidad por año.

In [17]:
import pandas as pd

camaras = pd.read_csv("../data/raw/Camaras/camaras_sdm_arcgis.csv")

camaras = camaras[camaras["ELEMENTO"] == "CÁMARA FOTODETECCIÓN"]
print(camaras.shape)

(93, 8)


In [18]:
camaras = camaras.rename(columns={"ID": "id_camara",
                                  "DIRECCIÓN_EXACTA_CÁMARA": "direccion",
                                  "LATITUD": "latitud",
                                  "LONGITUD": "longitud",
                                  "CORREDOR": "corredor",
                                  "LOCALIDAD": "localidad"})
camaras = camaras[["id_camara", "direccion", "latitud", "longitud", "corredor", "localidad"]]

print(camaras.isna().sum())
print(camaras["id_camara"].duplicated().sum())

id_camara    0
direccion    0
latitud      0
longitud     0
corredor     0
localidad    0
dtype: int64
0


In [19]:
camaras["localidad"] = camaras["localidad"].str.strip().str.upper()

camaras["sitio"] = camaras["id_camara"].str.split("/").str[0]
print(camaras["sitio"].nunique())

camaras["fuente"] = "Capa oficial SDM (feb 2024)"
camaras.head()

41


,id_camara,direccion,latitud,longitud,corredor,localidad,sitio,fuente
0,DEI-001/CAM-01,AV BOYACÁ - CL 35 SUR (N-S),4.615664,-74.140267,Av. Boyacá,KENNEDY,DEI-001,Capa oficial SDM (feb 2024)
1,DEI-001/CAM-02,AV BOYACÁ - CL 35 SUR (S-N),4.615398,-74.140261,Av. Boyacá,KENNEDY,DEI-001,Capa oficial SDM (feb 2024)
2,DEI-002/CAM-01,AV BOYACÁ - CL 5A (N-S),4.628593,-74.137980,Av. Boyacá,KENNEDY,DEI-002,Capa oficial SDM (feb 2024)
3,DEI-002/CAM-02,AV BOYACÁ - CL 5A (S-N),4.628479,-74.137915,Av. Boyacá,KENNEDY,DEI-002,Capa oficial SDM (feb 2024)
4,DEI-003/CAM-01,AV CL 68 - CL 4D (S-N),4.627311,-74.123054,Av. Carrera 68,PUENTE ARANDA,DEI-003,Capa oficial SDM (feb 2024)


In [20]:
nuevas = pd.read_csv("../data/raw/Camaras/camaras_nuevas.csv")
nuevas["sitio"] = nuevas["id_camara"]
camaras = pd.concat([camaras, nuevas], ignore_index=True)

camaras = camaras.drop_duplicates(subset="id_camara")
print(camaras.shape)

(102, 8)


In [21]:
camaras.to_csv("../data/processed/camaras.csv", index=False)

Población

In [22]:
poblacion = pd.read_excel("../data/raw/Poblacion/anexo-proyecciones-poblacion-bogota-desagreacion-loc-2018-2035-UPZ-2018-2024.xlsx",
                          sheet_name="Localidades_edad_quinquenal", header=11, dtype={"COD_LOC": str})

poblacion = poblacion[poblacion["AREA"] == "Total"]

poblacion = poblacion[["COD_LOC", "NOM_LOC", "AÑO", "TOTAL"]]
poblacion = poblacion.rename(columns={"COD_LOC": "cod_localidad", "NOM_LOC": "localidad",
                                      "AÑO": "anio", "TOTAL": "poblacion"})

poblacion = poblacion[poblacion["anio"] <= 2025]
print(poblacion.shape)

(160, 4)


Igualar los nombres de las localidades

In [23]:
poblacion["localidad"] = poblacion["localidad"].str.strip().str.upper()
poblacion["localidad"] = poblacion["localidad"].replace({"LA CANDELARIA": "CANDELARIA",
                                                         "CIUDAD BOLIVAR": "CIUDAD BOLÍVAR"})

siniestros = pd.read_csv("../data/processed/siniestros.csv.gz")
print(set(siniestros["Localidad"].dropna()) - set(poblacion["localidad"]))
print(set(poblacion["localidad"]) - set(siniestros["Localidad"].dropna()))

set()
set()


In [24]:
print(poblacion[["localidad", "anio"]].duplicated().sum())
poblacion.to_csv("../data/processed/poblacion.csv", index=False)

0


## Decisiones
- Cámaras: de 143 filas nos quedamos con las 93 cámaras (quitamos 50 gabinetes). No hay datos faltantes ni repetidos. Están en 41 sitios.
- Pasamos la localidad a mayúsculas para que sea igual a la de los siniestros.
- Agregamos las cámaras anunciadas en agosto de 2026 (camaras_nuevas.csv): 9 filas, una por punto y sentido, que suman 18 cámaras en 6 cruces. Sus coordenadas son la mediana de los siniestros del SIGAT en cada cruce. En total quedan 102 filas.
- Población: usamos el área total de cada localidad, de 2018 a 2025 (160 filas). El DANE no trae 2015-2017.
- Cambiamos "LA CANDELARIA" por "CANDELARIA" y "CIUDAD BOLIVAR" por "CIUDAD BOLÍVAR" para que coincidan con el SIGAT. Ya coinciden las 20 localidades.